# 🚗 Coleta de Dados — Mercado Automotivo Brasil

Este notebook coleta dados de duas fontes públicas e os organiza em subdiretórios dentro de `FONTE/`:

```
FONTE/
├── DENATRAN/          ← frota por município, UF, tipo de veículo (Base dos Dados)
│   ├── raw/           ← parquet bruto baixado do BigQuery
│   └── processed/     ← CSV limpo e pronto para análise
└── ANFAVEA/           ← produção, emplacamento e exportação por marca
    ├── raw/           ← Excel originais baixados do site
    └── processed/     ← CSV consolidado de todos os anos
```

**Fontes:**
- DENATRAN: `basedosdados.br_denatran_frota.municipio_tipo` via Base dos Dados
- ANFAVEA: https://anfavea.com.br/site/edicoes-em-excel/

## 0. Configuração e dependências

In [1]:
# Instalar dependências caso necessário
# !pip install basedosdados pandas requests openpyxl tqdm

In [2]:
import basedosdados as bd
import pandas as pd
import requests
import os
import time
from pathlib import Path
from tqdm.notebook import tqdm

# ─────────────────────────────────────────────
# ⚙️  CONFIGURAÇÕES — edite aqui
# ─────────────────────────────────────────────

BILLING_ID   = "pequisa-automovel-bd"   # ID do projeto no Google Cloud
BASE_DIR     = Path("FONTE")          # raiz dos dados (relativa ao notebook)

# Anos ANFAVEA a baixar (arquivos disponíveis a partir de 2012)
ANFAVEA_ANOS = list(range(2012, 2027))

# ─────────────────────────────────────────────
# Criar estrutura de diretórios
# ─────────────────────────────────────────────
dirs = [
    BASE_DIR / "DENATRAN" / "raw",
    BASE_DIR / "DENATRAN" / "processed",
    BASE_DIR / "ANFAVEA"  / "raw",
    BASE_DIR / "ANFAVEA"  / "processed",
]
for d in dirs:
    d.mkdir(parents=True, exist_ok=True)

print("✅ Estrutura de diretórios criada:")
for d in dirs:
    print(f"   {d}")

✅ Estrutura de diretórios criada:
   FONTE\DENATRAN\raw
   FONTE\DENATRAN\processed
   FONTE\ANFAVEA\raw
   FONTE\ANFAVEA\processed


---
## 1. DENATRAN — Frota por município e tipo de veículo
> **Fonte:** Base dos Dados / DENATRAN  
> **Cobertura:** mensal por município, UF, tipo de veículo  
> **Destino:** `FONTE/DENATRAN/`

In [3]:
DENATRAN_RAW  = BASE_DIR / "DENATRAN" / "raw"  / "denatran_municipio_tipo.parquet"
DENATRAN_CSV  = BASE_DIR / "DENATRAN" / "processed" / "denatran_municipio_tipo.csv"

query_denatran = """
  SELECT
    dados.ano              AS ano,
    dados.mes              AS mes,
    dados.sigla_uf         AS sigla_uf,
    dados.id_municipio     AS id_municipio,
    dir.nome               AS municipio_nome,
    dados.tipo_veiculo     AS tipo_veiculo,
    dados.quantidade       AS quantidade
  FROM `basedosdados.br_denatran_frota.municipio_tipo` AS dados
  LEFT JOIN (
    SELECT DISTINCT id_municipio, nome
    FROM `basedosdados.br_bd_diretorios_brasil.municipio`
  ) AS dir
    ON dados.id_municipio = dir.id_municipio
  ORDER BY ano, mes, sigla_uf, municipio_nome
"""

In [4]:
if DENATRAN_RAW.exists():
    print(f"⚡ Parquet já existe — carregando do cache: {DENATRAN_RAW}")
    df_denatran = pd.read_parquet(DENATRAN_RAW)
else:
    print("⬇️  Baixando DENATRAN do BigQuery (pode levar alguns minutos)...")
    df_denatran = bd.read_sql(query=query_denatran, billing_project_id=BILLING_ID)
    df_denatran.to_parquet(DENATRAN_RAW, index=False)
    print(f"✅ Salvo em: {DENATRAN_RAW}")

print(f"\n📊 Shape: {df_denatran.shape[0]:,} linhas × {df_denatran.shape[1]} colunas")
print(f"📅 Período: {df_denatran['ano'].min()} a {df_denatran['ano'].max()}")
print(f"🗺️  UFs: {df_denatran['sigla_uf'].nunique()} | Municípios: {df_denatran['municipio_nome'].nunique():,}")
df_denatran.head()

⬇️  Baixando DENATRAN do BigQuery (pode levar alguns minutos)...
Downloading: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████|
✅ Salvo em: FONTE\DENATRAN\raw\denatran_municipio_tipo.parquet

📊 Shape: 31,726,380 linhas × 7 colunas
📅 Período: 2003 a 2025
🗺️  UFs: 27 | Municípios: 5,297


,ano,mes,sigla_uf,id_municipio,municipio_nome,tipo_veiculo,quantidade
0,2003,1,AC,1200013,Acrelândia,side-car,0
1,2003,1,AC,1200013,Acrelândia,motoneta,15
2,2003,1,AC,1200013,Acrelândia,semi-reboque,2
3,2003,1,AC,1200013,Acrelândia,caminhao trator,1
4,2003,1,AC,1200013,Acrelândia,camioneta,36


In [5]:
# Salvar CSV processado
df_denatran.to_csv(DENATRAN_CSV, index=False, encoding="utf-8-sig")
print(f"✅ CSV salvo em: {DENATRAN_CSV}")
print(f"   Tamanho: {DENATRAN_CSV.stat().st_size / 1_048_576:.1f} MB")

✅ CSV salvo em: FONTE\DENATRAN\processed\denatran_municipio_tipo.csv
   Tamanho: 1393.8 MB


---
## 2. ANFAVEA — Produção, emplacamento e exportação por marca
> **Fonte:** anfavea.com.br — Dados Estatísticos para Download  
> **Cobertura:** anual (série mensal desde 1957), nacional, por empresa/marca  
> **Destino:** `FONTE/ANFAVEA/`

São baixados **3 arquivos por ano**:
| Arquivo | Conteúdo |
|---|---|
| `siteautoveiculos{ANO}.xlsx` | Produção, emplacamento, exportações totais |
| `emplacamentos_nacionais_{ANO}.xlsx` | Emplacamentos de nacionais por empresa e marca |
| `emplacamentos_importados_{ANO}.xlsx` | Emplacamentos de importados por empresa e marca |

In [6]:
# URLs dos arquivos ANFAVEA
# Padrão descoberto via inspeção do site anfavea.com.br/site/edicoes-em-excel/

def anfavea_urls(ano: int) -> list[dict]:
    base = "https://anfavea.com.br/docs"
    return [
        {
            "nome": f"autoveiculos_{ano}.xlsx",
            "url" : f"{base}/siteautoveiculos{ano}.xlsx",
            "tipo": "producao_emplacamento"
        },
        {
            "nome": f"emplacamentos_nacionais_{ano}.xlsx",
            "url" : f"{base}/emplacamentos_nacionais_{ano}.xlsx",
            "tipo": "emplacamentos_nacionais"
        },
        {
            "nome": f"emplacamentos_importados_{ano}.xlsx",
            "url" : f"{base}/emplacamentos_importados_{ano}.xlsx",
            "tipo": "emplacamentos_importados"
        },
    ]


def download_arquivo(url: str, destino: Path, timeout: int = 30) -> bool:
    """Baixa um arquivo com retry simples. Retorna True se OK."""
    if destino.exists():
        return True  # cache
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        resp = requests.get(url, headers=headers, timeout=timeout)
        if resp.status_code == 200 and len(resp.content) > 1000:
            destino.write_bytes(resp.content)
            return True
        return False
    except Exception:
        return False

In [7]:
ANFAVEA_RAW = BASE_DIR / "ANFAVEA" / "raw"
log = []  # registro de sucessos/falhas

for ano in tqdm(ANFAVEA_ANOS, desc="Anos ANFAVEA"):
    for arq in anfavea_urls(ano):
        destino = ANFAVEA_RAW / arq["nome"]
        ok = download_arquivo(arq["url"], destino)
        log.append({"ano": ano, "tipo": arq["tipo"], "arquivo": arq["nome"], "ok": ok})
        if not ok and not destino.exists():
            print(f"   ⚠️  Não encontrado: {arq['url']}")
        time.sleep(0.3)  # respeitar o servidor

df_log = pd.DataFrame(log)
total = len(df_log)
baixados = df_log['ok'].sum()
print(f"\n✅ {baixados}/{total} arquivos baixados com sucesso")
df_log[~df_log['ok']]  # mostrar os que falharam

Anos ANFAVEA:   0%|          | 0/15 [00:00<?, ?it/s]

   ⚠️  Não encontrado: https://anfavea.com.br/docs/siteautoveiculos2012.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_nacionais_2012.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_importados_2012.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/siteautoveiculos2013.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_nacionais_2013.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_importados_2013.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/siteautoveiculos2014.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_nacionais_2014.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_importados_2014.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/siteautoveiculos2015.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_nacionais_2015.xlsx
   ⚠️  Não encontrado: https://anfavea.com.br/docs/emplacamentos_importados_2015.xlsx
   ⚠️  N

,ano,tipo,arquivo,ok
0,2012,producao_emplacamento,autoveiculos_2012.xlsx,False
1,2012,emplacamentos_nacionais,emplacamentos_nacionais_2012.xlsx,False
2,2012,emplacamentos_importados,emplacamentos_importados_2012.xlsx,False
3,2013,producao_emplacamento,autoveiculos_2013.xlsx,False
4,2013,emplacamentos_nacionais,emplacamentos_nacionais_2013.xlsx,False
5,2013,emplacamentos_importados,emplacamentos_importados_2013.xlsx,False
6,2014,producao_emplacamento,autoveiculos_2014.xlsx,False
7,2014,emplacamentos_nacionais,emplacamentos_nacionais_2014.xlsx,False
8,2014,emplacamentos_importados,emplacamentos_importados_2014.xlsx,False
9,2015,producao_emplacamento,autoveiculos_2015.xlsx,False


In [8]:
# Listar arquivos baixados com tamanho
print("📁 Arquivos em FONTE/ANFAVEA/raw/:")
for f in sorted(ANFAVEA_RAW.glob("*.xlsx")):
    print(f"   {f.name:50s}  {f.stat().st_size/1024:6.0f} KB")

📁 Arquivos em FONTE/ANFAVEA/raw/:


### 2.1 Consolidar ANFAVEA — produção e emplacamento total

In [9]:
# O arquivo siteautoveiculos{ANO}.xlsx tem várias abas.
# Esta célula inspeciona as abas do arquivo mais recente disponível.

arquivos_totais = sorted(ANFAVEA_RAW.glob("autoveiculos_*.xlsx"))

if arquivos_totais:
    ultimo = arquivos_totais[-1]
    xl = pd.ExcelFile(ultimo)
    print(f"📋 Abas em '{ultimo.name}':")
    for aba in xl.sheet_names:
        print(f"   → {aba}")
else:
    print("⚠️  Nenhum arquivo autoveiculos_*.xlsx encontrado. Verifique os downloads.")

⚠️  Nenhum arquivo autoveiculos_*.xlsx encontrado. Verifique os downloads.


In [10]:
# ─────────────────────────────────────────────
# Ajuste ABA_PRINCIPAL após inspecionar a célula acima.
# Geralmente é algo como 'Autoveiculos' ou a primeira aba.
# ─────────────────────────────────────────────
ABA_PRINCIPAL = 0   # 0 = primeira aba; ou use o nome: ex. "Autoveiculos"

frames = []
for arq in tqdm(arquivos_totais, desc="Consolidando ANFAVEA"):
    try:
        ano = int(arq.stem.split("_")[-1])
        df_tmp = pd.read_excel(arq, sheet_name=ABA_PRINCIPAL, header=None)
        df_tmp.insert(0, "ano_arquivo", ano)
        df_tmp.insert(1, "fonte_arquivo", arq.name)
        frames.append(df_tmp)
    except Exception as e:
        print(f"   ⚠️  Erro em {arq.name}: {e}")

if frames:
    df_anfavea_raw = pd.concat(frames, ignore_index=True)
    saida_csv = BASE_DIR / "ANFAVEA" / "processed" / "anfavea_producao_emplacamento_raw.csv"
    df_anfavea_raw.to_csv(saida_csv, index=False, encoding="utf-8-sig")
    print(f"\n✅ Consolidado salvo: {saida_csv}")
    print(f"   Shape: {df_anfavea_raw.shape}")
    df_anfavea_raw.head(10)
else:
    print("⚠️  Nenhum frame consolidado. Verifique downloads e nome da aba.")

Consolidando ANFAVEA: 0it [00:00, ?it/s]

⚠️  Nenhum frame consolidado. Verifique downloads e nome da aba.


---
## 3. Resumo final

In [11]:
print("=" * 55)
print("  RESUMO DA COLETA")
print("=" * 55)

# DENATRAN
if DENATRAN_CSV.exists():
    n = pd.read_csv(DENATRAN_CSV, nrows=0).shape   # só cabeçalho
    size_mb = DENATRAN_CSV.stat().st_size / 1_048_576
    print(f"\n📂 DENATRAN")
    print(f"   {DENATRAN_CSV.name} — {size_mb:.1f} MB")

# ANFAVEA raw
xlsx_files = list((BASE_DIR / "ANFAVEA" / "raw").glob("*.xlsx"))
total_mb = sum(f.stat().st_size for f in xlsx_files) / 1_048_576
print(f"\n📂 ANFAVEA/raw")
print(f"   {len(xlsx_files)} arquivos Excel — {total_mb:.1f} MB total")

# ANFAVEA processed
proc_files = list((BASE_DIR / "ANFAVEA" / "processed").glob("*.csv"))
print(f"\n📂 ANFAVEA/processed")
for f in proc_files:
    print(f"   {f.name} — {f.stat().st_size/1_048_576:.1f} MB")

print("\n" + "=" * 55)
print("  Próximos passos:")
print("  1. Inspecionar abas dos XLSXs (célula 2.1)")
print("  2. Ajustar ABA_PRINCIPAL e re-consolidar")
print("  3. Cruzar DENATRAN (geo) × ANFAVEA (marca)")
print("=" * 55)

  RESUMO DA COLETA

📂 DENATRAN
   denatran_municipio_tipo.csv — 1393.8 MB

📂 ANFAVEA/raw
   0 arquivos Excel — 0.0 MB total

📂 ANFAVEA/processed

  Próximos passos:
  1. Inspecionar abas dos XLSXs (célula 2.1)
  2. Ajustar ABA_PRINCIPAL e re-consolidar
  3. Cruzar DENATRAN (geo) × ANFAVEA (marca)
